## Retrieval Augmented Generation (RAG)

## Indexing: Document loading and preprocessing

### `PyPDFLoader`

In [1]:
import copy

from langchain_community.document_loaders import PyPDFLoader

/Users/sebastiancg/Developer/scaceresg/ai-engineering/notebooks/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
loader_pdf = PyPDFLoader(file_path="data_files/Introduction_to_Data_and_Data_Science.pdf")
pages_pdf = loader_pdf.load()

In [3]:
pages_pdf[0]

Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2023-11-09T10:16:34+02:00', 'author': 'Hristina  Hristova', 'moddate': '2023-11-09T10:16:34+02:00', 'source': 'data_files/Introduction_to_Data_and_Data_Science.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='Analysis vs Analytics \nAlright! So… \nLet’s discuss the not-so-obvious differences \nbetween the terms analysis and analytics. \nDue to the similarity of the words, some people \nbelieve they share the same meaning, and thus \nuse them interchangeably. Technically, this \nisn’t correct. There is, in fact, a distinct \ndifference between the two. And the reason \nfor one often being used instead of the other \nis the lack of a transparent understanding \nof both. \nSo, let’s clear this up, shall we? \nFirst, we will start with analysis. \nConsider the following… \nYou have a huge dataset containing data of \nvarious types. Instead

In [4]:
print("length of pages_pdf: ", len(pages_pdf))
print("type of pages_pdf: ", type(pages_pdf))

length of pages_pdf:  6
type of pages_pdf:  <class 'list'>


We will remove new line separators `\n` to reduce token consumption:

In [5]:
pages_pdf_cut = copy.deepcopy(pages_pdf)

In [6]:
" ".join(pages_pdf_cut[0].page_content.split())

'Analysis vs Analytics Alright! So… Let’s discuss the not-so-obvious differences between the terms analysis and analytics. Due to the similarity of the words, some people believe they share the same meaning, and thus use them interchangeably. Technically, this isn’t correct. There is, in fact, a distinct difference between the two. And the reason for one often being used instead of the other is the lack of a transparent understanding of both. So, let’s clear this up, shall we? First, we will start with analysis. Consider the following… You have a huge dataset containing data of various types. Instead of tackling the entire dataset and running the risk of becoming overwhelmed, you separate it into easier to digest chunks and study them individually and examine how they relate to other parts. And that’s analysis in a nutshell. One important thing to remember, however, is that you perform analyses on things that have already happened in the past. Such as using an analysis to explain how a

In [7]:
for page in pages_pdf_cut:
    page.page_content = " ".join(page.page_content.split())

In [8]:
pages_pdf_cut

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2023-11-09T10:16:34+02:00', 'author': 'Hristina  Hristova', 'moddate': '2023-11-09T10:16:34+02:00', 'source': 'data_files/Introduction_to_Data_and_Data_Science.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='Analysis vs Analytics Alright! So… Let’s discuss the not-so-obvious differences between the terms analysis and analytics. Due to the similarity of the words, some people believe they share the same meaning, and thus use them interchangeably. Technically, this isn’t correct. There is, in fact, a distinct difference between the two. And the reason for one often being used instead of the other is the lack of a transparent understanding of both. So, let’s clear this up, shall we? First, we will start with analysis. Consider the following… You have a huge dataset containing data of various types. Instead of tackling the entire dataset

Using the https://platform.openai.com/tokenizer page:

* Tokens with `\n`: 421 tokens

* Tokens without `\n`: 306 tokens

### `Docx2txtLoader` and Text Splitters

In [14]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import CharacterTextSplitter

In [11]:
loader = Docx2txtLoader(file_path="data_files/Introduction_to_Data_and_Data_Science.docx")
pages_docx = loader.load()

for page in pages_docx:
    page.page_content = " ".join(page.page_content.split())

In [12]:
pages_docx[0].page_content

"# Introduction to Data and Data Science ## Analysis vs Analytics Alright! So… Let’s discuss the not-so-obvious differences between the terms analysis and analytics. Due to the similarity of the words, some people believe they share the same meaning, and thus use them interchangeably. Technically, this isn’t correct. There is, in fact, a distinct difference between the two. And the reason for one often being used instead of the other is the lack of a transparent understanding of both. So, let’s clear this up, shall we? First, we will start with analysis. Consider the following… You have a huge dataset containing data of various types. Instead of tackling the entire dataset and running the risk of becoming overwhelmed, you separate it into easier to digest chunks and study them individually and examine how they relate to other parts. And that’s analysis in a nutshell. One important thing to remember, however, is that you perform analyses on things that have already happened in the past.

In [13]:
len(pages_docx[0].page_content)

8305

#### Character Text Splitter

In [ ]:
character_splitter = CharacterTextSplitter(
    separator="",
    chunk_size=500,
    chunk_overlap=0, # No overlap = 0, experiment with the overlaps
)

In [16]:
pages_docx_split = character_splitter.split_documents(pages_docx)


In [ ]:
print(len(pages_docx_split)) # 8305 / 500 = 16.61 chunks ~= 17 chunks
print(len(pages_docx_split[-1].page_content)) # 0.61 * 500 = 305 tokens

17
305


In [ ]:
pages_docx_split[0].page_content

# Finishes with: ... transparent understanding of both. So, let’s c

'# Introduction to Data and Data Science ## Analysis vs Analytics Alright! So… Let’s discuss the not-so-obvious differences between the terms analysis and analytics. Due to the similarity of the words, some people believe they share the same meaning, and thus use them interchangeably. Technically, this isn’t correct. There is, in fact, a distinct difference between the two. And the reason for one often being used instead of the other is the lack of a transparent understanding of both. So, let’s c'

In [ ]:
pages_docx_split[1].page_content

# Starts with: ... lear this up, shall we? First,

'lear this up, shall we? First, we will start with analysis. Consider the following… You have a huge dataset containing data of various types. Instead of tackling the entire dataset and running the risk of becoming overwhelmed, you separate it into easier to digest chunks and study them individually and examine how they relate to other parts. And that’s analysis in a nutshell. One important thing to remember, however, is that you perform analyses on things that have already happened in the past.'

#### Markdown Header Text Splitter

In [22]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

In [41]:
loader = Docx2txtLoader(file_path="data_files/Introduction_to_Data_and_Data_Science.docx")
pages_docx = loader.load()

In [42]:
pages_docx

[Document(metadata={'source': 'data_files/Introduction_to_Data_and_Data_Science.docx'}, page_content="# Introduction to Data and Data Science\n\n## Analysis vs Analytics\n\nAlright! So…\nLet’s discuss the not-so-obvious differences\nbetween the terms analysis and analytics.\nDue to the similarity of the words, some people\nbelieve they share the same meaning, and thus\nuse them interchangeably. Technically, this\nisn’t correct. There is, in fact, a distinct\ndifference between the two. And the reason\nfor one often being used instead of the other\nis the lack of a transparent understanding\nof both.\nSo, let’s clear this up, shall we?\nFirst, we will start with analysis.\nConsider the following…\nYou have a huge dataset containing data of\nvarious types. Instead of tackling the entire\ndataset and running the risk of becoming overwhelmed,\nyou separate it into easier to digest chunks\nand study them individually and examine how\nthey relate to other parts. And that’s analysis\nin a nut

In [43]:
md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "Course Title"),
        ("##", "Lecture Title")
    ]
)
pages_md_split = md_splitter.split_text(pages_docx[0].page_content)

In [ ]:
pages_md_split
# Document(page_content, metadata)
# metadata: dict with the header information

[Document(metadata={'Course Title': 'Introduction to Data and Data Science', 'Lecture Title': 'Analysis vs Analytics'}, page_content="Alright! So…\nLet’s discuss the not-so-obvious differences\nbetween the terms analysis and analytics.\nDue to the similarity of the words, some people\nbelieve they share the same meaning, and thus\nuse them interchangeably. Technically, this\nisn’t correct. There is, in fact, a distinct\ndifference between the two. And the reason\nfor one often being used instead of the other\nis the lack of a transparent understanding\nof both.\nSo, let’s clear this up, shall we?\nFirst, we will start with analysis.\nConsider the following…\nYou have a huge dataset containing data of\nvarious types. Instead of tackling the entire\ndataset and running the risk of becoming overwhelmed,\nyou separate it into easier to digest chunks\nand study them individually and examine how\nthey relate to other parts. And that’s analysis\nin a nutshell.\nOne important thing to remember

In [45]:
pages_md_split[0].page_content

"Alright! So…\nLet’s discuss the not-so-obvious differences\nbetween the terms analysis and analytics.\nDue to the similarity of the words, some people\nbelieve they share the same meaning, and thus\nuse them interchangeably. Technically, this\nisn’t correct. There is, in fact, a distinct\ndifference between the two. And the reason\nfor one often being used instead of the other\nis the lack of a transparent understanding\nof both.\nSo, let’s clear this up, shall we?\nFirst, we will start with analysis.\nConsider the following…\nYou have a huge dataset containing data of\nvarious types. Instead of tackling the entire\ndataset and running the risk of becoming overwhelmed,\nyou separate it into easier to digest chunks\nand study them individually and examine how\nthey relate to other parts. And that’s analysis\nin a nutshell.\nOne important thing to remember, however,\nis that you perform analyses on things that\nhave already happened in the past. Such as\nusing an analysis to explain how

In [46]:
pages_md_split[0].metadata

{'Course Title': 'Introduction to Data and Data Science',
 'Lecture Title': 'Analysis vs Analytics'}

## Embeddings

In [1]:
import numpy as np

from dotenv import load_dotenv

from langchain_community.document_loaders import Docx2txtLoader
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter, MarkdownHeaderTextSplitter

load_dotenv()

/Users/sebastiancg/Developer/scaceresg/ai-engineering/notebooks/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
loader = Docx2txtLoader(file_path="data_files/Introduction_to_Data_and_Data_Science.docx")
pages_docx = loader.load()

# Split text into lectures using Markdown splitter
md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "Course Title"),
        ("##", "Lecture Title")
    ]
)
pages_md_split = md_splitter.split_text(pages_docx[0].page_content)

# Remove the new line separators
for page in pages_md_split:
    page.page_content = " ".join(page.page_content.split())

# Split text into chunks of 500 tokens with 50 tokens overlap (preserves the lecture structure)
character_splitter = CharacterTextSplitter(
    separator=".",
    chunk_size=500,
    chunk_overlap=50,
)

pages_char_split = character_splitter.split_documents(pages_md_split)

In [3]:
len(pages_char_split)

20

In [4]:
pages_char_split

[Document(metadata={'Course Title': 'Introduction to Data and Data Science', 'Lecture Title': 'Analysis vs Analytics'}, page_content='Alright! So… Let’s discuss the not-so-obvious differences between the terms analysis and analytics. Due to the similarity of the words, some people believe they share the same meaning, and thus use them interchangeably. Technically, this isn’t correct. There is, in fact, a distinct difference between the two. And the reason for one often being used instead of the other is the lack of a transparent understanding of both. So, let’s clear this up, shall we? First, we will start with analysis'),
 Document(metadata={'Course Title': 'Introduction to Data and Data Science', 'Lecture Title': 'Analysis vs Analytics'}, page_content='Consider the following… You have a huge dataset containing data of various types. Instead of tackling the entire dataset and running the risk of becoming overwhelmed, you separate it into easier to digest chunks and study them individu

Create embeddings client and use `embed_query()` on three strings:

* `pages_char_split[3]` # First lecture

* `pages_char_split[5]` # First lecture

* `pages_char_split[18]` # Second lecture (no relation to the first two!)

In [5]:
print(pages_char_split[3].page_content)
print(pages_char_split[3].metadata)

Analytics is essentially the application of logical and computational reasoning to the component parts obtained in an analysis. And in doing this you are looking for patterns and exploring what you could do with them in the future. Here, analytics branches off into two areas: qualitative analytics – this is using your intuition and experience in conjunction with the analysis to plan your next business move
{'Course Title': 'Introduction to Data and Data Science', 'Lecture Title': 'Analysis vs Analytics'}


In [6]:
print(pages_char_split[18].page_content)
print(pages_char_split[18].metadata)

More importantly, it will be sufficient for your need to create quick and accurate analyses. However, if your theoretical preparation is strong enough, you will find yourself restricted by software. Knowing a programming language such as R and Python, gives you the freedom to create specific, ad-hoc tools for each project you are working on
{'Course Title': 'Introduction to Data and Data Science', 'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need'}


In [7]:
embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")

In [8]:
vector_1 = embeddings.embed_query(pages_char_split[3].page_content)
vector_2 = embeddings.embed_query(pages_char_split[5].page_content)
vector_3 = embeddings.embed_query(pages_char_split[18].page_content)

In [9]:
vector_1

[0.003742626868188381,
 0.010484526865184307,
 0.010070833377540112,
 -0.03043227642774582,
 -0.01968919113278389,
 0.012074658647179604,
 -0.024239812046289444,
 -0.013018394820392132,
 0.010562093928456306,
 -0.027096876874566078,
 0.004783323034644127,
 0.01674162968993187,
 -0.013115353882312775,
 0.006341134663671255,
 -0.010898219421505928,
 -0.01657356694340706,
 0.03604298457503319,
 -0.0008976810495369136,
 0.020270947366952896,
 -0.02367098443210125,
 -0.04374801367521286,
 0.022494545206427574,
 -0.00870047602802515,
 -0.027407146990299225,
 -0.009999730624258518,
 0.0037006111815571785,
 0.010258288122713566,
 -0.0250284131616354,
 0.004815642721951008,
 -0.01821541041135788,
 0.009734708815813065,
 0.005672115832567215,
 -0.008002369664609432,
 0.0004969161236658692,
 -0.009055993519723415,
 0.0029475612100213766,
 0.011143849231302738,
 0.01587546057999134,
 0.015293705277144909,
 0.017633654177188873,
 0.01521613821387291,
 0.0009308087755925953,
 -0.03405208885669708,
 

In [10]:
print(len(vector_1))
print(len(vector_2))
print(len(vector_3))

1536
1536
1536


Compute the dot product:

In [11]:
print(np.dot(vector_1, vector_2)) # 0.8791284497943928
print(np.dot(vector_1, vector_3)) # 0.8000235828747095
print(np.dot(vector_2, vector_3)) # 0.7934993700101874

0.8791284497943928
0.8000235828747095
0.7934993700101874


Check the magnitude of vectors:

In [12]:
print(np.linalg.norm(vector_1))
print(np.linalg.norm(vector_2))
print(np.linalg.norm(vector_3))

0.9999999518969221
0.9999999432048748
0.9999999688261215


## Vector Database

* Let's finish the embeddings for all 20 text splits (from previous section)

* We will use the `OpenAIEmbeddings` class and an embedding model

* And store these chunks in a vector store

* LangChain supports integration with many vector stores: We'll use **Chroma**

In [13]:
from dotenv import load_dotenv

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

load_dotenv()

True

In [14]:
len(pages_char_split)

20

In [15]:
embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")

In [16]:
vector_store = Chroma.from_documents(
    documents=pages_char_split, # A list of the 20 text splits
    embedding=embeddings, # Embedding class
    persist_directory="vector_store" # Local directory to save the vector store
)

Let's "load" the vector store created in the `vector_store` directory:

In [17]:
vector_store_from_dir = Chroma(
    persist_directory="vector_store", # Local directory where the vector store is saved
    embedding_function=embeddings # Embedding class
)

In [ ]:
vector_store_from_dir.get()

{'ids': ['51dc4fb3-5fff-4186-8df0-8baf5dffc566',
  '44033f1e-e91b-4a2e-80ba-2a60d877f45a',
  'b8e90e56-240a-4fec-9b08-508d3f692b72',
  '9cc8034f-3e12-498e-9093-b699e7ac1d4b',
  'f8c6bfc7-965d-45e7-8995-7ab9be23a247',
  'faec7f37-fe6d-4617-ae76-566d555cac1e',
  'fcaf8365-d211-49a6-91b8-70a3ba50eeb7',
  '2b27aec4-4d99-4d7f-b030-433c74bb2c55',
  '468f8a23-c8a8-4d80-9505-2a57386d02fd',
  '3f9186c0-ea2c-4f55-a08c-b7d4ec20cc87',
  '46afc268-1133-4738-b981-0ddd2714a99e',
  '4504e4d4-9a23-4ffd-8b13-fab725bbe93a',
  '1e068741-d1c9-4369-a4e5-16713e8e38e9',
  '38efa5ed-c363-459c-86c0-b1a4a4b870f2',
  '23caf425-795e-4cca-8e1d-faadeb2e777d',
  'f9dd44a4-6b66-4f1e-88ef-544aea6630fd',
  '99c62bc9-3cb4-4f4f-ab1e-7dcdead4c206',
  '934b8f40-5bfd-4d7d-af73-57f9823680cc',
  '8779bfe1-daf1-4cbe-bd27-27cdfd6215e2',
  'afe9b8e2-bc34-4e7c-b896-8db6a395f8b3',
  '8b9a9bf7-f9b4-47b8-9a91-3841e4ee05d1',
  'eaca94d6-c952-44d7-8e91-baa393bbb806',
  '22db01be-da8f-44f8-80aa-7cff3bd54e8d',
  '17def8e5-3019-44c3-a58a-

In [19]:
vector_store_from_dir.get(
    ids="51dc4fb3-5fff-4186-8df0-8baf5dffc566",
    include=["metadatas", "embeddings"]
)

{'ids': ['51dc4fb3-5fff-4186-8df0-8baf5dffc566'],
 'embeddings': array([[ 0.00478017, -0.01535145,  0.02508651, ...,  0.02121745,
         -0.01364157, -0.00687695]], shape=(1, 1536)),
 'documents': None,
 'uris': None,
 'included': ['metadatas', 'embeddings'],
 'data': None,
 'metadatas': [{'Lecture Title': 'Analysis vs Analytics',
   'Course Title': 'Introduction to Data and Data Science'}]}

## Retrieval

**Task**:

* Define a question related to data science

* Create a vector representation of this question

* Retrieve a pre-selected number of documents relevant to this question

In [20]:
from dotenv import load_dotenv

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

load_dotenv()

True

In [21]:
embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")

In [22]:
vector_store_from_dir = Chroma(
    persist_directory="vector_store", # Local directory where the vector store is saved
    embedding_function=embeddings # Embedding class
)

In [ ]:
question = "What programming languages do data scientists use?"
# Expected response: R, Python, SQL, MATLAB, Java, JavaScript, C, C++, Scala (from the second lecture)

In [24]:
retrieved_docs = vector_store_from_dir.similarity_search(
    query=question,
    k=5 # Number of documents to retrieve
)

In [25]:
len(retrieved_docs)
retrieved_docs[0]

Document(id='1e068741-d1c9-4369-a4e5-16713e8e38e9', metadata={'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need', 'Course Title': 'Introduction to Data and Data Science'}, page_content='What about big data? Apart from R and Python, people working in this area are often proficient in other languages like Java or Scala. These two have not been developed specifically for doing statistical analyses, however they turn out to be very useful when combining data from multiple sources. All right! Let’s finish off with machine learning. When it comes to machine learning, we often deal with big data')

In [28]:
for doc in retrieved_docs:
    print("="*100)
    print("Page content: ", doc.page_content)
    print("Lecture title: ", doc.metadata["Lecture Title"])

Page content:  What about big data? Apart from R and Python, people working in this area are often proficient in other languages like Java or Scala. These two have not been developed specifically for doing statistical analyses, however they turn out to be very useful when combining data from multiple sources. All right! Let’s finish off with machine learning. When it comes to machine learning, we often deal with big data
Lecture title:  Programming Languages & Software Employed in Data Science - All the Tools You Need
Page content:  What about big data? Apart from R and Python, people working in this area are often proficient in other languages like Java or Scala. These two have not been developed specifically for doing statistical analyses, however they turn out to be very useful when combining data from multiple sources. All right! Let’s finish off with machine learning. When it comes to machine learning, we often deal with big data
Lecture title:  Programming Languages & Software Em

### Maximum Marginal Relevance (MMR) Search

This algorithm aims to balance between:

* Fetching documents relevant to the user's query (similarity score)

* Retrieving documents that are less redundant to those already retrieved (redundancy score)

The redundancy score is found by calculating its similarity with all retrieved documents and then taking the highest among these values.

* $Redudancy = \max{(similarity(retrieved)})$

* A higher number indicates that the document is less likely to contribute with new retrieved information

* $Diversity = - \max{(similarity(retrieved)})$

The linear combination between **relevance** (similarity) and **diversity** (the inverse of redundancy) is called **Marginal Relevance** [(Carbonnell & Goldstein, 1998)](https://dl.acm.org/doi/epdf/10.1145/290941.291025):

$$
\text{Marginal Relevance} = \lambda\cdot \text{similarity} - (1 - \lambda)\cdot \text{diversity}
$$

where $\lambda$ is between $0,1$

* $\lambda \approx 0$ diminishes the similarity component and gives more emphasis on diversity

* $\lambda \approx 1$ reduces the diversity, concentrating on similar results 

In [29]:
question = "What software do data scientists use?"
# Expected response: Excel, SPSS, Apache Hadoop, Apache HBase, MongoDB, PowerBI, SAS, Qlik, Tableau, EViews, Stata

In [30]:
retrieved_docs = vector_store_from_dir.similarity_search(
    query=question,
    k=3 # Number of documents to retrieve
)

In [31]:
for doc in retrieved_docs:
    print("="*100)
    print("Page content: ", doc.page_content)
    print("Lecture title: ", doc.metadata["Lecture Title"])

Page content:  As you can see from the infographic, R, and Python are the two most popular tools across all columns. Their biggest advantage is that they can manipulate data and are integrated within multiple data and data science software platforms. They are not just suitable for mathematical and statistical computations. In other words, R, and Python are adaptable. They can solve a wide variety of business and data-related problems from beginning to the end
Lecture title:  Programming Languages & Software Employed in Data Science - All the Tools You Need
Page content:  As you can see from the infographic, R, and Python are the two most popular tools across all columns. Their biggest advantage is that they can manipulate data and are integrated within multiple data and data science software platforms. They are not just suitable for mathematical and statistical computations. In other words, R, and Python are adaptable. They can solve a wide variety of business and data-related problems

Using the MMR:

In [32]:
retrieved_docs = vector_store_from_dir.max_marginal_relevance_search(
    query=question,
    k=3, # Number of documents to retrieve
    lambda_mult=0.1 # Lambda value (0, 1)
)

In [33]:
for doc in retrieved_docs:
    print("="*100)
    print("Page content: ", doc.page_content)
    print("Lecture title: ", doc.metadata["Lecture Title"])

Page content:  As you can see from the infographic, R, and Python are the two most popular tools across all columns. Their biggest advantage is that they can manipulate data and are integrated within multiple data and data science software platforms. They are not just suitable for mathematical and statistical computations. In other words, R, and Python are adaptable. They can solve a wide variety of business and data-related problems from beginning to the end
Lecture title:  Programming Languages & Software Employed in Data Science - All the Tools You Need
Page content:  Among the many applications we have plotted, we can say there is an increasing amount of software designed for working with big data such as Apache Hadoop, Apache Hbase, and Mongo DB. In terms of big data, Hadoop is the name that must stick with you. Hadoop is listed as a software in the sense that it is a collection of programs, but don’t imagine it as a nice-looking application
Lecture title:  Programming Languages &

In [34]:
retrieved_docs = vector_store_from_dir.max_marginal_relevance_search(
    query=question,
    k=3, # Number of documents to retrieve
    lambda_mult=0.1, # Lambda value (0, 1)
    filter={"Lecture Title": "Programming Languages & Software Employed in Data Science - All the Tools You Need"}
)

In [35]:
for doc in retrieved_docs:
    print("="*100)
    print("Page content: ", doc.page_content)
    print("Lecture title: ", doc.metadata["Lecture Title"])

Page content:  As you can see from the infographic, R, and Python are the two most popular tools across all columns. Their biggest advantage is that they can manipulate data and are integrated within multiple data and data science software platforms. They are not just suitable for mathematical and statistical computations. In other words, R, and Python are adaptable. They can solve a wide variety of business and data-related problems from beginning to the end
Lecture title:  Programming Languages & Software Employed in Data Science - All the Tools You Need
Page content:  Among the many applications we have plotted, we can say there is an increasing amount of software designed for working with big data such as Apache Hadoop, Apache Hbase, and Mongo DB. In terms of big data, Hadoop is the name that must stick with you. Hadoop is listed as a software in the sense that it is a collection of programs, but don’t imagine it as a nice-looking application
Lecture title:  Programming Languages &

In [36]:
retrieved_docs = vector_store_from_dir.max_marginal_relevance_search(
    query=question,
    k=3, # Number of documents to retrieve
    lambda_mult=0.8, # Lambda value (0, 1)
    filter={"Lecture Title": "Programming Languages & Software Employed in Data Science - All the Tools You Need"}
)

In [37]:
for doc in retrieved_docs:
    print("="*100)
    print("Page content: ", doc.page_content)
    print("Lecture title: ", doc.metadata["Lecture Title"])

Page content:  As you can see from the infographic, R, and Python are the two most popular tools across all columns. Their biggest advantage is that they can manipulate data and are integrated within multiple data and data science software platforms. They are not just suitable for mathematical and statistical computations. In other words, R, and Python are adaptable. They can solve a wide variety of business and data-related problems from beginning to the end
Lecture title:  Programming Languages & Software Employed in Data Science - All the Tools You Need
Page content:  Alright! So… How are the techniques used in data, business intelligence, or predictive analytics applied in real life? Certainly, with the help of computers. You can basically split the relevant tools into two categories—programming languages and software. Knowing a programming language enables you to devise programs that can execute specific operations. Moreover, you can reuse these programs whenever you need to execu

## Generation

In [38]:
from dotenv import load_dotenv

from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables import RunnableParallel
from langchain_openai import ChatOpenAI
from langchain_openai.embeddings import OpenAIEmbeddings


load_dotenv()

True

In [39]:
vector_store = Chroma(
    persist_directory="vector_store",
    embedding_function=OpenAIEmbeddings(model="text-embedding-ada-002")
)

In [40]:
# Vector store retriever
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,
        "lambda_mult": 0.1
    }
)

In [41]:
# Prompt template
TEMPLATE = """
Answer the following question:
{question}

To answer the question, use only the following context:
{context}

At the end of the response, specify the name of the lecture this context is taken from in the format:
Resources: *Lecture Title*
where *Lecture Title* is the title of all resource lectures.
"""

# Prompt template
prompt_template = PromptTemplate.from_template(TEMPLATE)

In [42]:
chat = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.1,
    max_completion_tokens=250,
    model_kwargs={
        "seed": 42,
    }
)

/Users/sebastiancg/Developer/scaceresg/ai-engineering/notebooks/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3687: UserWarning: Parameters {'seed'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  if await self.run_code(code, result, async_=asy):


In [43]:
question = "What software do data scientists use?"

In [48]:
rag_chain = {
    "context": retriever,
    "question": RunnablePassthrough(),
} | prompt_template

In [51]:
print(rag_chain.invoke(question).text)


Answer the following question:
What software do data scientists use?

To answer the question, use only the following context:
[Document(id='3f9186c0-ea2c-4f55-a08c-b7d4ec20cc87', metadata={'Course Title': 'Introduction to Data and Data Science', 'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need'}, page_content='As you can see from the infographic, R, and Python are the two most popular tools across all columns. Their biggest advantage is that they can manipulate data and are integrated within multiple data and data science software platforms. They are not just suitable for mathematical and statistical computations. In other words, R, and Python are adaptable. They can solve a wide variety of business and data-related problems from beginning to the end'), Document(id='f9dd44a4-6b66-4f1e-88ef-544aea6630fd', metadata={'Course Title': 'Introduction to Data and Data Science', 'Lecture Title': 'Programming Languages & Software Employed in D

In [58]:
rag_chain_final = (
    {
        "context": retriever,
        "question": RunnablePassthrough(),
    }
    | prompt_template
    | chat
    | StrOutputParser()
)

In [60]:
print(rag_chain_final.invoke(question))

Data scientists commonly use R and Python as their primary programming languages due to their ability to manipulate data and their integration with various data science software platforms. These languages are versatile and can address a wide range of business and data-related challenges. Additionally, for big data applications, tools like Apache Hadoop, Apache Hbase, and MongoDB are also utilized, with Hadoop being particularly notable as a collection of programs designed for big data processing.

Resources: Programming Languages & Software Employed in Data Science - All the Tools You Need
